In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

MIMIC_DIR = Path(
    "../data/raw/mimic-iii/"
    "physionet.org/files/mimiciii-demo/1.4"
)

OUTPUT_DIR = Path("../data/processed/mimic3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MIMIC:", MIMIC_DIR)
print("Output:", OUTPUT_DIR)

MIMIC: ../data/raw/mimic-iii/physionet.org/files/mimiciii-demo/1.4
Output: ../data/processed/mimic3


In [2]:
patients = pd.read_csv(MIMIC_DIR / "PATIENTS.csv")
admissions = pd.read_csv(MIMIC_DIR / "ADMISSIONS.csv")
icustays = pd.read_csv(MIMIC_DIR / "ICUSTAYS.csv")
diagnoses = pd.read_csv(MIMIC_DIR / "DIAGNOSES_ICD.csv")
d_icd = pd.read_csv(MIMIC_DIR / "D_ICD_DIAGNOSES.csv")
d_items = pd.read_csv(MIMIC_DIR / "D_ITEMS.csv")

# Attach ICD descriptions
diagnoses_full = diagnoses.merge(
    d_icd[["icd9_code", "short_title", "long_title"]],
    on="icd9_code",
    how="left"
)

# Identify sepsis diagnoses
sepsis_dx = diagnoses_full[
    diagnoses_full["long_title"]
    .fillna("")
    .str.contains(
        "sepsis|septicemia|septic shock",
        case=False,
        na=False
    )
].copy()

# ICU cohort
sepsis_stays = icustays[
    icustays["subject_id"].isin(sepsis_dx["subject_id"])
    & icustays["hadm_id"].isin(sepsis_dx["hadm_id"])
].copy()

sepsis_stays["intime"] = pd.to_datetime(sepsis_stays["intime"])
sepsis_stays["outtime"] = pd.to_datetime(sepsis_stays["outtime"])

print("Sepsis patients:", sepsis_stays["subject_id"].nunique())
print("Sepsis admissions:", sepsis_stays["hadm_id"].nunique())
print("Sepsis ICU stays:", sepsis_stays["icustay_id"].nunique())

Sepsis patients: 26
Sepsis admissions: 38
Sepsis ICU stays: 41


In [3]:
SEARCH_TERMS = {
    "heart_rate": "heart rate",
    "resp_rate": "respiratory rate",
    "map": "mean arterial pressure",
    "oxygen_saturation": "oxygen saturation",
    "temperature": "temperature",
    "gcs_eye": "gcs - eye",
    "gcs_verbal": "gcs - verbal",
    "gcs_motor": "gcs - motor",
}

item_lookup = {}

for feature, term in SEARCH_TERMS.items():

    matches = d_items[
        d_items["label"]
        .fillna("")
        .str.contains(
            term,
            case=False,
            na=False
        )
    ][
        ["itemid", "label", "abbreviation", "unitname", "param_type"]
    ]

    item_lookup[feature] = matches

    print("=" * 70)
    print(feature.upper())
    print(matches.to_string(index=False))

HEART_RATE
 itemid                   label    abbreviation unitname param_type
    211              Heart Rate             NaN      NaN        NaN
   3494       Lowest Heart Rate             NaN      NaN        NaN
 220045              Heart Rate              HR      bpm    Numeric
 220046 Heart rate Alarm - High HR Alarm - High      bpm    Numeric
 220047  Heart Rate Alarm - Low  HR Alarm - Low      bpm    Numeric
RESP_RATE
 itemid                          label                   abbreviation unitname param_type
    618               Respiratory Rate                            NaN      NaN        NaN
    619           Respiratory Rate Set                            NaN      NaN        NaN
 220210               Respiratory Rate                             RR insp/min    Numeric
 224688         Respiratory Rate (Set)         Respiratory Rate (Set) insp/min    Numeric
 224689 Respiratory Rate (spontaneous) Respiratory Rate (spontaneous) insp/min    Numeric
 224690       Respiratory Rate 

In [4]:
CLINICAL_ITEMS = {
    "heart_rate": [...],
    "map": [...],
    "resp_rate": [...],
    "oxygen_saturation": [...],
    "temperature": [...],
    "gcs_eye": [...],
    "gcs_verbal": [...],
    "gcs_motor": [...],
}

TARGET_ITEMIDS = [
    itemid
    for ids in CLINICAL_ITEMS.values()
    for itemid in ids
]

print("Total selected item IDs:", len(TARGET_ITEMIDS))

Total selected item IDs: 8


In [5]:
chartevents_path = MIMIC_DIR / "CHARTEVENTS.csv"

columns = [
    "subject_id",
    "hadm_id",
    "icustay_id",
    "charttime",
    "itemid",
    "value",
    "valuenum",
    "valueuom",
]

chartevents = pd.read_csv(
    chartevents_path,
    usecols=columns,
    low_memory=False
)

print("CHARTEVENTS loaded:", chartevents.shape)

CHARTEVENTS loaded: (758355, 8)


In [6]:
chartevents = chartevents[
    chartevents["icustay_id"].isin(
        sepsis_stays["icustay_id"]
    )
    &
    chartevents["itemid"].isin(
        TARGET_ITEMIDS
    )
].copy()

chartevents["charttime"] = pd.to_datetime(
    chartevents["charttime"]
)

print("Filtered clinical events:", chartevents.shape)

Filtered clinical events: (0, 8)


In [7]:
# Check the ID types and actual values

print("ICUSTAYS icustay_id:")
print(sepsis_stays["icustay_id"].dtype)
print(sepsis_stays["icustay_id"].head(10).tolist())

print("\nCHARTEVENTS icustay_id:")
print(chartevents["icustay_id"].dtype)
print(chartevents["icustay_id"].head(10).tolist())

print("\nCHARTEVENTS itemid dtype:")
print(chartevents["itemid"].dtype)

print("\nSelected item IDs:")
print(TARGET_ITEMIDS)

ICUSTAYS icustay_id:
int64
[206504, 264446, 228977, 226055, 227834, 235482, 203766, 285789, 248755, 223177]

CHARTEVENTS icustay_id:
float64
[]

CHARTEVENTS itemid dtype:
int64

Selected item IDs:
[Ellipsis, Ellipsis, Ellipsis, Ellipsis, Ellipsis, Ellipsis, Ellipsis, Ellipsis]


In [8]:
# Normalize IDs before filtering

sepsis_stays["icustay_id_clean"] = pd.to_numeric(
    sepsis_stays["icustay_id"],
    errors="coerce"
).astype("Int64")

chartevents["icustay_id_clean"] = pd.to_numeric(
    chartevents["icustay_id"],
    errors="coerce"
).astype("Int64")

chartevents["itemid_clean"] = pd.to_numeric(
    chartevents["itemid"],
    errors="coerce"
).astype("Int64")

sepsis_stay_ids = set(
    sepsis_stays["icustay_id_clean"].dropna()
)

print("Sepsis ICU IDs:", len(sepsis_stay_ids))

print(
    "Matching ICU IDs:",
    chartevents["icustay_id_clean"]
    .isin(sepsis_stay_ids)
    .sum()
)

Sepsis ICU IDs: 41
Matching ICU IDs: 0


In [9]:
matching_events = chartevents[
    chartevents["icustay_id_clean"].isin(
        sepsis_stay_ids
    )
]

print(
    "Events from our 41 ICU stays:",
    len(matching_events)
)

print(
    "Matching ICU stays:",
    matching_events["icustay_id_clean"].nunique()
)

Events from our 41 ICU stays: 0
Matching ICU stays: 0


In [10]:
map_matches = d_items[
    d_items["label"]
    .fillna("")
    .str.contains(
        "mean|arterial|blood pressure|pressure",
        case=False,
        na=False
    )
]

display(
    map_matches[
        [
            "itemid",
            "label",
            "abbreviation",
            "unitname",
            "param_type"
        ]
    ].head(100)
)

,itemid,label,abbreviation,unitname,param_type
7,1449,Arterial BP(Rad),NaN,NaN,NaN
56,51,Arterial BP [Systolic],NaN,NaN,NaN
57,52,Arterial BP Mean,NaN,NaN,NaN
58,53,Arterial Pressure,NaN,NaN,NaN
143,141,Cuff Pressure-Airway,NaN,NaN,NaN
...,...,...,...,...,...
3638,2562,INTRA ABD PRESSURE,NaN,NaN,NaN
3691,2647,art mean,NaN,NaN,NaN
3734,2704,ABDOMINAL PRESSURE,NaN,NaN,NaN
3744,2732,Femoral ABP (Mean),NaN,NaN,NaN


In [11]:
# ---------------------------------------------------------
# CHECK CHARTEVENTS → SEPSIS ICU LINK USING HADm_ID
# ---------------------------------------------------------

chartevents["charttime"] = pd.to_datetime(
    chartevents["charttime"],
    errors="coerce"
)

# Keep only patients/admissions belonging to our cohort
cohort_keys = sepsis_stays[
    ["subject_id", "hadm_id"]
].drop_duplicates()

clinical_events = chartevents.merge(
    cohort_keys,
    on=["subject_id", "hadm_id"],
    how="inner"
)

print(
    "CHARTEVENTS belonging to sepsis admissions:",
    len(clinical_events)
)

print(
    "Unique patients:",
    clinical_events["subject_id"].nunique()
)

print(
    "Unique admissions:",
    clinical_events["hadm_id"].nunique()
)

CHARTEVENTS belonging to sepsis admissions: 0
Unique patients: 0
Unique admissions: 0


In [12]:
# ---------------------------------------------------------
# ASSIGN EVENTS TO ICU STAYS USING TIME
# ---------------------------------------------------------

clinical_events = clinical_events.merge(
    sepsis_stays[
        [
            "subject_id",
            "hadm_id",
            "icustay_id",
            "intime",
            "outtime"
        ]
    ],
    on=["subject_id", "hadm_id"],
    how="inner",
    suffixes=("", "_icu")
)

clinical_events = clinical_events[
    (clinical_events["charttime"] >= clinical_events["intime"])
    &
    (clinical_events["charttime"] <= clinical_events["outtime"])
].copy()

print(
    "Clinical events inside ICU windows:",
    len(clinical_events)
)

print(
    "ICU stays with clinical events:",
    clinical_events["icustay_id"].nunique()
)

print(
    "Patients with clinical events:",
    clinical_events["subject_id"].nunique()
)

Clinical events inside ICU windows: 0
ICU stays with clinical events: 0
Patients with clinical events: 0


In [13]:
# ============================================================
# DIAGNOSE MIMIC-III CHARTEVENTS ↔ SEPSIS COHORT LINK
# ============================================================

print("SEPSIS STAYS")
print(sepsis_stays[["subject_id", "hadm_id", "icustay_id"]].head(10))
print(sepsis_stays[["subject_id", "hadm_id"]].dtypes)

print("\nCHARTEVENTS")
print(chartevents[["subject_id", "hadm_id", "icustay_id"]].head(10))
print(chartevents[["subject_id", "hadm_id"]].dtypes)

print("\nCHARTEVENTS NULL COUNTS")
print(
    chartevents[
        ["subject_id", "hadm_id", "icustay_id"]
    ].isna().sum()
)

# Normalize IDs to strings for a completely safe comparison
sepsis_subjects = set(
    sepsis_stays["subject_id"]
    .astype(str)
    .str.strip()
)

sepsis_hadm = set(
    sepsis_stays["hadm_id"]
    .astype(str)
    .str.strip()
)

chartevent_subjects = set(
    chartevents["subject_id"]
    .dropna()
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

chartevent_hadm = set(
    chartevents["hadm_id"]
    .dropna()
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

print("\n================================================")
print("SEPSIS SUBJECTS:", len(sepsis_subjects))
print("CHART SUBJECTS:", len(chartevent_subjects))
print(
    "OVERLAPPING SUBJECTS:",
    len(sepsis_subjects & chartevent_subjects)
)

print("\nSEPSIS HADM:", len(sepsis_hadm))
print("CHART HADM:", len(chartevent_hadm))
print(
    "OVERLAPPING HADM:",
    len(sepsis_hadm & chartevent_hadm)
)

print("\nExample sepsis subject IDs:")
print(list(sepsis_subjects)[:10])

print("\nExample CHARTEVENTS subject IDs:")
print(list(chartevent_subjects)[:10])

print("\nExample sepsis admission IDs:")
print(list(sepsis_hadm)[:10])

print("\nExample CHARTEVENTS admission IDs:")
print(list(chartevent_hadm)[:10])

SEPSIS STAYS
    subject_id  hadm_id  icustay_id
0        10006   142345      206504
2        10013   165520      264446
4        10019   177759      228977
7        10029   132349      226055
11       10036   189483      227834
12       10038   111115      235482
17       10045   126949      203766
19       10056   100375      285789
21       10059   122098      248755
22       10061   145203      223177
subject_id    int64
hadm_id       int64
dtype: object

CHARTEVENTS
Empty DataFrame
Columns: [subject_id, hadm_id, icustay_id]
Index: []
subject_id    int64
hadm_id       int64
dtype: object

CHARTEVENTS NULL COUNTS
subject_id    0
hadm_id       0
icustay_id    0
dtype: int64

SEPSIS SUBJECTS: 26
CHART SUBJECTS: 0
OVERLAPPING SUBJECTS: 0

SEPSIS HADM: 38
CHART HADM: 0
OVERLAPPING HADM: 0

Example sepsis subject IDs:
['10056', '10061', '40177', '10013', '40655', '10038', '41976', '41914', '10059', '10045']

Example CHARTEVENTS subject IDs:
[]

Example sepsis admission IDs:
['132349', '1

In [14]:
chartevents_raw = pd.read_csv(
    MIMIC_DIR / "CHARTEVENTS.csv",
    usecols=[
        "subject_id",
        "hadm_id",
        "icustay_id",
        "charttime",
        "itemid",
        "value",
        "valuenum",
        "valueuom"
    ],
    low_memory=False
)

print("Raw CHARTEVENTS:", chartevents_raw.shape)

print(
    chartevents_raw[
        ["subject_id", "hadm_id", "icustay_id"]
    ].head()
)

Raw CHARTEVENTS: (758355, 8)
   subject_id  hadm_id  icustay_id
0       40124   126179    279554.0
1       40124   126179    279554.0
2       40124   126179    279554.0
3       40124   126179    279554.0
4       40124   126179    279554.0


In [15]:
cohort_keys = sepsis_stays[
    ["subject_id", "hadm_id"]
].drop_duplicates()

test = chartevents_raw.merge(
    cohort_keys,
    on=["subject_id", "hadm_id"],
    how="inner"
)

print("Events belonging to sepsis admissions:", len(test))
print("Patients:", test["subject_id"].nunique())
print("Admissions:", test["hadm_id"].nunique())

Events belonging to sepsis admissions: 267006
Patients: 25
Admissions: 35


In [17]:
for itemid in [
    211, 220045,
    618, 220210, 224689, 224690,
    52, 220052, 220181,
    646, 220277,
    184, 220739,
    723, 223900,
    454, 223901
]:
    row = d_items[d_items["itemid"] == itemid]

    if len(row):
        print("\n", itemid)
        print(
            row[
                ["label", "abbreviation", "unitname", "param_type"]
            ].to_string(index=False)
        )


 211
     label abbreviation unitname param_type
Heart Rate          NaN      NaN        NaN

 220045
     label abbreviation unitname param_type
Heart Rate           HR      bpm    Numeric

 618
           label abbreviation unitname param_type
Respiratory Rate          NaN      NaN        NaN

 220210
           label abbreviation unitname param_type
Respiratory Rate           RR insp/min    Numeric

 224689
                         label                   abbreviation unitname param_type
Respiratory Rate (spontaneous) Respiratory Rate (spontaneous) insp/min    Numeric

 224690
                   label             abbreviation unitname param_type
Respiratory Rate (Total) Respiratory Rate (Total) insp/min    Numeric

 52
           label abbreviation unitname param_type
Arterial BP Mean          NaN      NaN        NaN

 220052
                       label abbreviation unitname param_type
Arterial Blood Pressure mean         ABPm     mmHg    Numeric

 220181
                         

In [20]:
# ============================================================
# CLEAN MIMIC-III CLINICAL EXTRACTION
# ============================================================

# Start from untouched raw CHARTEVENTS
events = chartevents_raw.copy()

events["charttime"] = pd.to_datetime(
    events["charttime"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Clinical item IDs
# ------------------------------------------------------------

CLINICAL_ITEMS = {
    "heart_rate": [211, 220045],

    "resp_rate": [
        618,
        220210,
        224689,
        224690
    ],

    "map": [
        52,
        220052,
        220181
    ],

    "spo2": [
        646,
        220277
    ],

    "gcs_eye": [
        184,
        220739
    ],

    "gcs_verbal": [
        723,
        223900
    ],

    "gcs_motor": [
        454,
        223901
    ]
}

item_to_feature = {
    itemid: feature
    for feature, ids in CLINICAL_ITEMS.items()
    for itemid in ids
}

TARGET_ITEMIDS = list(item_to_feature.keys())

print("Selected item IDs:", TARGET_ITEMIDS)

# ------------------------------------------------------------
# 2. Keep only sepsis admissions
# ------------------------------------------------------------

cohort_keys = sepsis_stays[
    ["subject_id", "hadm_id"]
].drop_duplicates()

events = events.merge(
    cohort_keys,
    on=["subject_id", "hadm_id"],
    how="inner"
)

print("Events in sepsis admissions:", len(events))

# ------------------------------------------------------------
# 3. Select only our clinical variables
# ------------------------------------------------------------

events = events[
    events["itemid"].isin(TARGET_ITEMIDS)
].copy()

print("Selected clinical events:", len(events))

# ------------------------------------------------------------
# 4. IMPORTANT:
# Remove CHARTEVENTS icustay_id before merging ICU table
# ------------------------------------------------------------

events = events.drop(
    columns=["icustay_id"],
    errors="ignore"
)

# ------------------------------------------------------------
# 5. Add ICU stay information
# ------------------------------------------------------------

icu_info = sepsis_stays[
    [
        "subject_id",
        "hadm_id",
        "icustay_id",
        "intime",
        "outtime"
    ]
].copy()

icu_info["intime"] = pd.to_datetime(
    icu_info["intime"]
)

icu_info["outtime"] = pd.to_datetime(
    icu_info["outtime"]
)

events = events.merge(
    icu_info,
    on=["subject_id", "hadm_id"],
    how="inner"
)

# ------------------------------------------------------------
# 6. Keep events occurring during ICU stay
# ------------------------------------------------------------

events = events[
    (events["charttime"] >= events["intime"]) &
    (events["charttime"] <= events["outtime"])
].copy()

print("Events inside ICU windows:", len(events))
print(
    "ICU stays:",
    events["icustay_id"].nunique()
)
print(
    "Patients:",
    events["subject_id"].nunique()
)

# ------------------------------------------------------------
# 7. Map item IDs → feature names
# ------------------------------------------------------------

events["feature"] = events["itemid"].map(
    item_to_feature
)

# ------------------------------------------------------------
# 8. Convert numeric values
# ------------------------------------------------------------

events["valuenum"] = pd.to_numeric(
    events["valuenum"],
    errors="coerce"
)

# ------------------------------------------------------------
# 9. Relative ICU time
# ------------------------------------------------------------

events["hours_from_icu_admission"] = (
    events["charttime"] - events["intime"]
).dt.total_seconds() / 3600

events["hour"] = (
    events["hours_from_icu_admission"]
    .floordiv(1)
    .astype(int)
)

# ------------------------------------------------------------
# 10. Basic physiological range filtering
# ------------------------------------------------------------

valid_ranges = {
    "heart_rate": (20, 250),
    "resp_rate": (3, 80),
    "map": (20, 180),
    "spo2": (50, 100),
    "gcs_eye": (1, 4),
    "gcs_verbal": (1, 5),
    "gcs_motor": (1, 6)
}

for feature, (low, high) in valid_ranges.items():

    mask = events["feature"] == feature

    events.loc[
        mask &
        (
            (events["valuenum"] < low) |
            (events["valuenum"] > high)
        ),
        "valuenum"
    ] = np.nan

# Keep observations with usable numeric values
events = events[
    events["valuenum"].notna()
].copy()

print("Usable numeric events:", len(events))

# ------------------------------------------------------------
# 11. Save event-level clinical data
# ------------------------------------------------------------

events.to_csv(
    OUTPUT_DIR / "clinical_events.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "clinical_events.csv"
)

Selected item IDs: [211, 220045, 618, 220210, 224689, 224690, 52, 220052, 220181, 646, 220277, 184, 220739, 723, 223900, 454, 223901]
Events in sepsis admissions: 267006
Selected clinical events: 24057
Events inside ICU windows: 23863
ICU stays: 38
Patients: 25
Usable numeric events: 23618
Saved: ../data/processed/mimic3/clinical_events.csv


In [21]:
# ============================================================
# RESET CLINICAL EXTRACTION
# ============================================================

# Reload raw CHARTEVENTS
chartevents_raw = pd.read_csv(
    MIMIC_DIR / "CHARTEVENTS.csv",
    usecols=[
        "subject_id",
        "hadm_id",
        "icustay_id",
        "charttime",
        "itemid",
        "value",
        "valuenum",
        "valueuom"
    ],
    low_memory=False
)

chartevents_raw["charttime"] = pd.to_datetime(
    chartevents_raw["charttime"],
    errors="coerce"
)

print("Raw CHARTEVENTS:", chartevents_raw.shape)

# Make sure ICU timestamps are datetime
sepsis_stays["intime"] = pd.to_datetime(
    sepsis_stays["intime"]
)

sepsis_stays["outtime"] = pd.to_datetime(
    sepsis_stays["outtime"]
)

print("Sepsis ICU stays:", sepsis_stays["icustay_id"].nunique())

Raw CHARTEVENTS: (758355, 8)
Sepsis ICU stays: 41


In [22]:
# ============================================================
# HOURLY CLINICAL TRAJECTORY
# ============================================================

hourly = (
    events
    .pivot_table(
        index=[
            "subject_id",
            "hadm_id",
            "icustay_id",
            "hour"
        ],
        columns="feature",
        values="valuenum",
        aggfunc="mean"
    )
    .reset_index()
)

hourly.columns.name = None

# Make sure all expected features exist
for feature in [
    "heart_rate",
    "resp_rate",
    "map",
    "spo2",
    "gcs_eye",
    "gcs_verbal",
    "gcs_motor"
]:
    if feature not in hourly.columns:
        hourly[feature] = np.nan

# ------------------------------------------------------------
# GCS
# ------------------------------------------------------------

hourly["gcs_total"] = (
    hourly["gcs_eye"]
    + hourly["gcs_verbal"]
    + hourly["gcs_motor"]
)

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

hourly = hourly.sort_values(
    ["icustay_id", "hour"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Previous observed GCS
# ------------------------------------------------------------

hourly["previous_gcs"] = (
    hourly
    .groupby("icustay_id")["gcs_total"]
    .shift(1)
)

# ------------------------------------------------------------
# GCS change
# ------------------------------------------------------------

hourly["gcs_change"] = (
    hourly["gcs_total"]
    - hourly["previous_gcs"]
)

print("Hourly dataset:", hourly.shape)

display(hourly.head(10))

Hourly dataset: (4556, 14)


,subject_id,hadm_id,icustay_id,hour,gcs_eye,gcs_motor,gcs_verbal,heart_rate,map,resp_rate,spo2,gcs_total,previous_gcs,gcs_change
0,10076,198503,201006,0,NaN,NaN,NaN,100.000000,NaN,32.000000,93.000000,NaN,NaN,NaN
1,10076,198503,201006,1,NaN,NaN,NaN,104.500000,NaN,33.500000,94.000000,NaN,NaN,NaN
2,10076,198503,201006,2,NaN,NaN,NaN,104.500000,NaN,32.500000,99.500000,NaN,NaN,NaN
3,10076,198503,201006,3,4.0,6.0,5.0,103.000000,NaN,36.000000,99.000000,15.0,NaN,NaN
4,10076,198503,201006,4,NaN,NaN,NaN,90.000000,NaN,32.000000,99.000000,NaN,15.0,NaN
5,10076,198503,201006,5,NaN,NaN,NaN,103.000000,NaN,35.000000,97.000000,NaN,NaN,NaN
6,10076,198503,201006,6,NaN,NaN,NaN,90.000000,NaN,35.000000,99.000000,NaN,NaN,NaN
7,10076,198503,201006,7,NaN,NaN,NaN,101.000000,NaN,25.000000,93.000000,NaN,NaN,NaN
8,10076,198503,201006,8,NaN,NaN,NaN,107.000000,NaN,33.000000,90.500000,NaN,NaN,NaN
9,10076,198503,201006,9,NaN,NaN,NaN,74.666667,NaN,22.333333,97.333333,NaN,NaN,NaN


In [23]:
# ============================================================
# FIX GCS TRAJECTORY
# ============================================================

hourly = hourly.sort_values(
    ["icustay_id", "hour"]
).reset_index(drop=True)

# GCS is only observed at certain times.
# Carry the last observed GCS forward to represent
# the patient's most recently known neurological state.

hourly["gcs_last_observed"] = (
    hourly
    .groupby("icustay_id")["gcs_total"]
    .ffill()
)

# Previous observed GCS BEFORE the current observation
hourly["previous_observed_gcs"] = (
    hourly
    .groupby("icustay_id")["gcs_last_observed"]
    .shift(1)
)

# Only calculate a change when a NEW GCS was actually observed
hourly["gcs_change"] = np.where(
    hourly["gcs_total"].notna(),
    hourly["gcs_total"] - hourly["previous_observed_gcs"],
    np.nan
)

print("GCS observations:", hourly["gcs_total"].notna().sum())

print(
    "GCS deterioration events (<= -3):",
    (hourly["gcs_change"] <= -3).sum()
)

display(
    hourly.loc[
        hourly["gcs_change"] <= -3,
        [
            "subject_id",
            "icustay_id",
            "hour",
            "previous_observed_gcs",
            "gcs_total",
            "gcs_change"
        ]
    ].head(20)
)

GCS observations: 1209
GCS deterioration events (<= -3): 35


,subject_id,icustay_id,hour,previous_observed_gcs,gcs_total,gcs_change
11,10076,201006,11,15.0,8.0,-7.0
16,10076,201006,16,8.0,3.0,-5.0
169,10045,203766,20,15.0,3.0,-12.0
289,10045,203766,140,9.0,6.0,-3.0
290,10045,203766,141,6.0,3.0,-3.0
320,41976,205170,27,14.0,11.0,-3.0
356,41976,205170,63,15.0,12.0,-3.0
428,41976,209797,45,11.0,6.0,-5.0
697,10124,222779,54,15.0,10.0,-5.0
761,10061,223177,36,11.0,7.0,-4.0


In [24]:
# ============================================================
# NEUROLOGICAL DETERIORATION EVENT
# ============================================================

hourly["sae"] = (
    hourly["gcs_change"] <= -3
).astype(int)

print("Total deterioration events:", hourly["sae"].sum())

print(
    "ICU stays with deterioration:",
    hourly.loc[
        hourly["sae"] == 1,
        "icustay_id"
    ].nunique()
)

print(
    "Patients with deterioration:",
    hourly.loc[
        hourly["sae"] == 1,
        "subject_id"
    ].nunique()
)

Total deterioration events: 35
ICU stays with deterioration: 16
Patients with deterioration: 12


In [25]:
display(
    hourly[
        hourly["sae"] == 1
    ][
        [
            "subject_id",
            "icustay_id",
            "hour",
            "previous_observed_gcs",
            "gcs_total",
            "gcs_change"
        ]
    ]
)

,subject_id,icustay_id,hour,previous_observed_gcs,gcs_total,gcs_change
11,10076,201006,11,15.0,8.0,-7.0
16,10076,201006,16,8.0,3.0,-5.0
169,10045,203766,20,15.0,3.0,-12.0
289,10045,203766,140,9.0,6.0,-3.0
290,10045,203766,141,6.0,3.0,-3.0
320,41976,205170,27,14.0,11.0,-3.0
356,41976,205170,63,15.0,12.0,-3.0
428,41976,209797,45,11.0,6.0,-5.0
697,10124,222779,54,15.0,10.0,-5.0
761,10061,223177,36,11.0,7.0,-4.0


In [26]:
print("MAP observations:", hourly["map"].notna().sum())

print(
    hourly["map"].describe()
)

display(
    hourly[
        hourly["map"].notna()
    ][
        [
            "subject_id",
            "icustay_id",
            "hour",
            "map"
        ]
    ].head(20)
)

MAP observations: 3635
count    3635.000000
mean       71.951206
std        12.333543
min        26.000000
25%        63.000000
50%        71.000000
75%        79.500000
max       130.000000
Name: map, dtype: float64


,subject_id,icustay_id,hour,map
11,10076,201006,11,75.000000
12,10076,201006,12,75.000000
13,10076,201006,13,80.500000
14,10076,201006,14,75.000000
15,10076,201006,15,81.000000
16,10076,201006,16,60.000000
17,10076,201006,17,73.000000
18,10076,201006,18,75.000000
19,10076,201006,19,69.000000
20,10076,201006,20,61.000000


In [27]:
print(
    d_items[
        d_items["itemid"].isin(
            [52, 220052, 220181]
        )
    ][
        [
            "itemid",
            "label",
            "abbreviation",
            "unitname",
            "param_type"
        ]
    ]
)

      itemid                             label abbreviation unitname  \
57        52                  Arterial BP Mean          NaN      NaN   
9530  220052      Arterial Blood Pressure mean         ABPm     mmHg   
9548  220181  Non Invasive Blood Pressure mean         NBPm     mmHg   

     param_type  
57          NaN  
9530    Numeric  
9548    Numeric  


In [28]:
hourly.to_csv(
    OUTPUT_DIR / "clinical_hourly_v1.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "clinical_hourly_v1.csv"
)

Saved: ../data/processed/mimic3/clinical_hourly_v1.csv
